# Multi-Conflict Data Collection

This notebook expands the project from a Syria-only case study into a two-conflict comparative dataset.

## Current Scope

The current implementation includes two conflict cases:

- Syrian Civil War
- Russia–Ukraine War

The purpose is to compare **neighboring host countries** with **non-neighbor major host countries** across different war contexts.

The output of this notebook is a combined host-country-year panel dataset for 2010–2024.

Final output file:

`data/processed/conflict_host_panel_2010_2024.csv`

In [1]:
from pathlib import Path
import zipfile
import time
import io
import requests

import pandas as pd
import numpy as np

YEAR_FROM = 2010
YEAR_TO = 2024

REPO_ROOT = Path("..")
RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
OUTPUTS_DIR = REPO_ROOT / "outputs"
FIG_DIR = OUTPUTS_DIR / "figures"
TABLE_DIR = OUTPUTS_DIR / "tables"

for p in [RAW_DIR, PROCESSED_DIR, FIG_DIR, TABLE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

UCDP_BRD_CONFLICT_ZIP_URL = "https://ucdp.uu.se/downloads/brd/ucdp-brd-conf-251-csv.zip"
UNHCR_POPULATION_URL = "https://api.unhcr.org/population/v1/population/"
WORLD_BANK_V2_BASE = "https://api.worldbank.org/v2"

## Helper Functions

This section defines small helper functions for downloading data from online sources.

The retry logic is useful because APIs can sometimes fail temporarily or return rate-limit errors.

In [2]:
def _request_with_retries(method, url, *, params=None, headers=None, timeout=60, max_retries=5, backoff=2.0):
    attempt = 0

    while True:
        attempt += 1

        try:
            resp = requests.request(
                method,
                url,
                params=params,
                headers=headers,
                timeout=timeout
            )

            if resp.status_code in (429, 500, 502, 503, 504):
                if attempt >= max_retries:
                    resp.raise_for_status()
                time.sleep(backoff ** (attempt - 1))
                continue

            resp.raise_for_status()
            return resp

        except Exception:
            if attempt >= max_retries:
                raise
            time.sleep(backoff ** (attempt - 1))


def download_binary(url, path):
    response = _request_with_retries("GET", url, timeout=120)
    path.write_bytes(response.content)
    return path

## Conflict Case Definitions

Each conflict case has:

- an origin country
- a conflict start year used to create `post_conflict`
- a set of host countries
- a host-country group

The `host_group` variable is important because the project compares neighboring host countries with non-neighbor host countries.

In [3]:
CONFLICT_CASES = [
    {
        "conflict_case": "Syrian Civil War",
        "origin_name": "Syria",
        "origin_iso": "SYR",
        "ucdp_location": "syria",
        "post_conflict_start": 2011,
        "hosts": {
            "TUR": {"name": "Turkey", "group": "neighboring_host"},
            "LBN": {"name": "Lebanon", "group": "neighboring_host"},
            "JOR": {"name": "Jordan", "group": "neighboring_host"},
            "IRQ": {"name": "Iraq", "group": "neighboring_host"},
            "EGY": {"name": "Egypt", "group": "regional_non_neighbor_host"},
            "DEU": {"name": "Germany", "group": "non_neighbor_european_host"},
            "SWE": {"name": "Sweden", "group": "non_neighbor_european_host"},
            "AUT": {"name": "Austria", "group": "non_neighbor_european_host"},
            "NLD": {"name": "Netherlands", "group": "non_neighbor_european_host"},
        }
    },
    {
        "conflict_case": "Russia-Ukraine War",
        "origin_name": "Ukraine",
        "origin_iso": "UKR",
        "ucdp_location": "ukraine",
        "post_conflict_start": 2022,
        "hosts": {
            "POL": {"name": "Poland", "group": "neighboring_host"},
            "ROU": {"name": "Romania", "group": "neighboring_host"},
            "MDA": {"name": "Moldova", "group": "neighboring_host"},
            "HUN": {"name": "Hungary", "group": "neighboring_host"},
            "SVK": {"name": "Slovakia", "group": "neighboring_host"},
            "DEU": {"name": "Germany", "group": "non_neighbor_european_host"},
            "AUT": {"name": "Austria", "group": "non_neighbor_european_host"},
            "NLD": {"name": "Netherlands", "group": "non_neighbor_european_host"},
            "SWE": {"name": "Sweden", "group": "non_neighbor_european_host"},
            "CZE": {"name": "Czechia", "group": "non_neighbor_european_host"},
            "ITA": {"name": "Italy", "group": "non_neighbor_european_host"},
            "ESP": {"name": "Spain", "group": "non_neighbor_european_host"},
            "FRA": {"name": "France", "group": "non_neighbor_european_host"},
        }
    }
]

for case in CONFLICT_CASES:
    print(case["conflict_case"], "-", len(case["hosts"]), "host countries")

Syrian Civil War - 9 host countries
Russia-Ukraine War - 13 host countries


## Conflict Intensity Data

This section downloads the UCDP Battle-Related Deaths Dataset.

For each conflict case, annual conflict intensity is measured using the total annual `bd_best` value for the origin-country conflict location.

In [4]:
ucdp_zip_path = RAW_DIR / "ucdp_brd_conf_251.zip"

if not ucdp_zip_path.exists():
    download_binary(UCDP_BRD_CONFLICT_ZIP_URL, ucdp_zip_path)

with zipfile.ZipFile(ucdp_zip_path, "r") as zf:
    csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]

    if not csv_names:
        raise ValueError("UCDP zip does not contain a CSV file.")

    ucdp_csv_name = sorted(csv_names)[0]

    with zf.open(ucdp_csv_name) as f:
        ucdp = pd.read_csv(f)

ucdp.columns = [c.strip() for c in ucdp.columns]

print("UCDP shape:", ucdp.shape)
print("UCDP columns:", ucdp.columns.tolist())
ucdp.head()

UCDP shape: (1586, 25)
UCDP columns: ['conflict_id', 'dyad_id', 'location_inc', 'side_a', 'side_a_id', 'side_a_2nd', 'side_b', 'side_b_id', 'side_b_2nd', 'incompatibility', 'territory_name', 'year', 'bd_best', 'bd_low', 'bd_high', 'type_of_conflict', 'battle_location', 'gwno_a', 'gwno_a_2nd', 'gwno_b', 'gwno_b_2nd', 'gwno_loc', 'gwno_battle', 'region', 'version']


,conflict_id,dyad_id,location_inc,side_a,side_a_id,side_a_2nd,side_b,side_b_id,side_b_2nd,incompatibility,...,type_of_conflict,battle_location,gwno_a,gwno_a_2nd,gwno_b,gwno_b_2nd,gwno_loc,gwno_battle,region,version
0,11447,10006,Ethiopia,Government of Ethiopia,97,NaN,IGLF,1115,NaN,1,...,3,Ethiopia,530,NaN,NaN,NaN,530,530,4,25.1
1,372,10054,Mali,Government of Mali,72,NaN,FPLA,934,NaN,1,...,3,Mali,432,NaN,NaN,NaN,432,432,4,25.1
2,372,11985,Mali,Government of Mali,72,NaN,CMA,1158,NaN,1,...,3,Mali,432,NaN,NaN,NaN,432,432,4,25.1
3,372,11985,Mali,Government of Mali,72,NaN,CMA,1158,NaN,1,...,3,Mali,432,NaN,NaN,NaN,432,432,4,25.1
4,372,11985,Mali,Government of Mali,72,"Government of Armenia, Government of Austria, ...",CMA,1158,NaN,1,...,4,Mali,432,"371, 305, 771, 211, 434, 760, 346, 439, 516, 8...",NaN,NaN,432,432,4,25.1


In [5]:
def build_conflict_intensity(case):
    origin_iso = case["origin_iso"]
    location = case["ucdp_location"]

    case_ucdp = ucdp[
        (ucdp["year"] >= YEAR_FROM) &
        (ucdp["year"] <= YEAR_TO) &
        (ucdp["location_inc"].astype(str).str.strip().str.lower() == location)
    ].copy()

    intensity = (
        case_ucdp.groupby("year", as_index=False)["bd_best"]
        .sum()
        .rename(columns={"bd_best": "conflict_intensity"})
    )

    year_df = pd.DataFrame({"year": list(range(YEAR_FROM, YEAR_TO + 1))})

    intensity = year_df.merge(intensity, on="year", how="left")
    intensity["conflict_intensity"] = intensity["conflict_intensity"].fillna(0)
    intensity["origin_iso"] = origin_iso
    intensity["conflict_case"] = case["conflict_case"]

    return intensity


conflict_intensity_all = pd.concat(
    [build_conflict_intensity(case) for case in CONFLICT_CASES],
    ignore_index=True
)

print("Conflict intensity shape:", conflict_intensity_all.shape)
conflict_intensity_all.head(30)

Conflict intensity shape: (30, 4)


,year,conflict_intensity,origin_iso,conflict_case
0,2010,0.0,SYR,Syrian Civil War
1,2011,1203.0,SYR,Syrian Civil War
2,2012,50490.0,SYR,Syrian Civil War
3,2013,72016.0,SYR,Syrian Civil War
4,2014,65345.0,SYR,Syrian Civil War
5,2015,50181.0,SYR,Syrian Civil War
6,2016,43650.0,SYR,Syrian Civil War
7,2017,24126.0,SYR,Syrian Civil War
8,2018,13072.0,SYR,Syrian Civil War
9,2019,7379.0,SYR,Syrian Civil War


## Refugee Exposure Data

This section downloads refugee stock data from the UNHCR Refugee Statistics API.

The main refugee exposure variables are:

- `refugee_stock`
- `asylum_seekers`
- `refugees_per_1000`
- `log_refugee_stock`

The variable `refugees_per_1000` is calculated later after merging with World Bank population data.

In [6]:
def fetch_unhcr_population(case):
    origin_iso = case["origin_iso"]
    host_iso_list = list(case["hosts"].keys())

    params = {
        "download": "true",
        "cf_type": "ISO",
        "yearFrom": str(YEAR_FROM),
        "yearTo": str(YEAR_TO),
        "coo": origin_iso,
        "coa": ",".join(host_iso_list),
        "limit": "10000",
    }

    resp = _request_with_retries("GET", UNHCR_POPULATION_URL, params=params)

    base_name = (
        f"unhcr_population_coo_{origin_iso}_hosts_"
        f"{'_'.join(host_iso_list)}_{YEAR_FROM}_{YEAR_TO}"
    )

    if resp.content[:2] == b"PK":
        zip_path = RAW_DIR / f"{base_name}.zip"
        zip_path.write_bytes(resp.content)

        with zipfile.ZipFile(io.BytesIO(resp.content), "r") as zf:
            csv_names = [n for n in zf.namelist() if n.lower().endswith(".csv")]
            xlsx_names = [
                n for n in zf.namelist()
                if n.lower().endswith(".xlsx") or n.lower().endswith(".xls")
            ]

            if csv_names:
                with zf.open(csv_names[0]) as f:
                    unhcr = pd.read_csv(f)
            elif xlsx_names:
                with zf.open(xlsx_names[0]) as f:
                    unhcr = pd.read_excel(io.BytesIO(f.read()))
            else:
                raise ValueError("UNHCR zip did not contain a readable CSV or Excel file.")
    else:
        csv_path = RAW_DIR / f"{base_name}.csv"
        csv_path.write_bytes(resp.content)
        unhcr = pd.read_csv(csv_path)

    unhcr.columns = [c.strip() for c in unhcr.columns]

    needed_cols = [
        "Year",
        "Country of origin (ISO)",
        "Country of asylum (ISO)",
        "Refugees under UNHCR's mandate",
        "Asylum-seekers"
    ]

    missing_cols = [c for c in needed_cols if c not in unhcr.columns]
    if missing_cols:
        raise ValueError(f"Missing UNHCR columns for {case['conflict_case']}: {missing_cols}")

    small = unhcr[needed_cols].copy()

    small = small.rename(columns={
        "Year": "year",
        "Country of origin (ISO)": "origin_iso",
        "Country of asylum (ISO)": "host_iso",
        "Refugees under UNHCR's mandate": "refugee_stock",
        "Asylum-seekers": "asylum_seekers"
    })

    small["year"] = pd.to_numeric(small["year"], errors="coerce").astype("Int64")
    small["origin_iso"] = small["origin_iso"].astype(str).str.upper().str.strip()
    small["host_iso"] = small["host_iso"].astype(str).str.upper().str.strip()
    small["refugee_stock"] = pd.to_numeric(small["refugee_stock"], errors="coerce")
    small["asylum_seekers"] = pd.to_numeric(small["asylum_seekers"], errors="coerce")

    small = small[
        (small["origin_iso"] == origin_iso) &
        (small["host_iso"].isin(host_iso_list)) &
        (small["year"] >= YEAR_FROM) &
        (small["year"] <= YEAR_TO)
    ].copy()

    small["conflict_case"] = case["conflict_case"]

    return small.sort_values(["host_iso", "year"]).reset_index(drop=True)

In [7]:
unhcr_frames = []

for case in CONFLICT_CASES:
    print("Fetching UNHCR data for:", case["conflict_case"])
    case_unhcr = fetch_unhcr_population(case)
    print(case_unhcr.shape)
    unhcr_frames.append(case_unhcr)

unhcr_all = pd.concat(unhcr_frames, ignore_index=True)

print("Combined UNHCR shape:", unhcr_all.shape)
print(unhcr_all.head())
print(unhcr_all.tail())

Fetching UNHCR data for: Syrian Civil War
(135, 6)
Fetching UNHCR data for: Russia-Ukraine War
(189, 6)
Combined UNHCR shape: (324, 6)
   year origin_iso host_iso  refugee_stock  asylum_seekers     conflict_case
0  2010        SYR      AUT            505             471  Syrian Civil War
1  2011        SYR      AUT            903             456  Syrian Civil War
2  2012        SYR      AUT           1675             694  Syrian Civil War
3  2013        SYR      AUT           2748            1561  Syrian Civil War
4  2014        SYR      AUT           6653               0  Syrian Civil War
     year origin_iso host_iso  refugee_stock  asylum_seekers  \
319  2020        UKR      SWE            232             390   
320  2021        UKR      SWE            226             334   
321  2022        UKR      SWE          46857             989   
322  2023        UKR      SWE          43467            1111   
323  2024        UKR      SWE          45993             151   

          conflict

## Macroeconomic Data

This section downloads host-country macroeconomic indicators from the World Bank World Development Indicators API.

Indicators used:

- GDP growth
- inflation
- unemployment
- trade as percentage of GDP
- current account balance as percentage of GDP
- population

These indicators are collected for all host countries in both conflict cases.

In [8]:
def worldbank_fetch_one(countries_iso3, indicator, year_from, year_to, per_page=20000):
    rows = []

    countries_seg = ";".join([c.lower() for c in countries_iso3])
    url = f"{WORLD_BANK_V2_BASE}/country/{countries_seg}/indicator/{indicator}"

    params = {
        "format": "json",
        "date": f"{year_from}:{year_to}",
        "per_page": per_page,
        "page": 1,
        "source": 2,
    }

    while True:
        resp = _request_with_retries(
            "GET",
            url,
            params=params,
            headers={"Accept": "application/json"},
            timeout=30,
            max_retries=3,
            backoff=1.5
        )

        payload = resp.json()

        if not isinstance(payload, list) or len(payload) < 2:
            raise ValueError(f"Unexpected World Bank response for {indicator}")

        meta, data = payload[0], payload[1]

        if data is None:
            break

        for item in data:
            host_iso = item.get("countryiso3code")
            year_str = item.get("date")
            val = item.get("value", None)

            if host_iso is None or year_str is None:
                continue

            try:
                year = int(year_str)
            except Exception:
                continue

            rows.append({
                "host_iso": host_iso,
                "year": year,
                "indicator_code": indicator,
                "value": val
            })

        pages = int(meta.get("pages", 1))
        page = int(meta.get("page", 1))

        if page >= pages:
            break

        params["page"] = page + 1

    return pd.DataFrame(rows)

In [9]:
ALL_HOSTS = sorted(
    set(
        host_iso
        for case in CONFLICT_CASES
        for host_iso in case["hosts"].keys()
    )
)

ALL_WDI = [
    "NY.GDP.MKTP.KD.ZG",    # GDP growth
    "FP.CPI.TOTL.ZG",      # Inflation
    "SL.UEM.TOTL.ZS",      # Unemployment
    "NE.TRD.GNFS.ZS",      # Trade as % of GDP
    "BN.CAB.XOKA.GD.ZS",   # Current account balance as % of GDP
    "SP.POP.TOTL",         # Population
]

wdi_frames = []

for indicator in ALL_WDI:
    print("Fetching World Bank indicator:", indicator)
    df_ind = worldbank_fetch_one(ALL_HOSTS, indicator, YEAR_FROM, YEAR_TO)
    wdi_frames.append(df_ind)

wdi_tidy = pd.concat(wdi_frames, ignore_index=True)

print("WDI tidy shape:", wdi_tidy.shape)
print("Countries:", sorted(wdi_tidy["host_iso"].dropna().unique()))
print("Years:", wdi_tidy["year"].min(), "-", wdi_tidy["year"].max())
wdi_tidy.head()

Fetching World Bank indicator: NY.GDP.MKTP.KD.ZG
Fetching World Bank indicator: FP.CPI.TOTL.ZG
Fetching World Bank indicator: SL.UEM.TOTL.ZS
Fetching World Bank indicator: NE.TRD.GNFS.ZS
Fetching World Bank indicator: BN.CAB.XOKA.GD.ZS
Fetching World Bank indicator: SP.POP.TOTL
WDI tidy shape: (1620, 4)
Countries: ['AUT', 'CZE', 'DEU', 'EGY', 'ESP', 'FRA', 'HUN', 'IRQ', 'ITA', 'JOR', 'LBN', 'MDA', 'NLD', 'POL', 'ROU', 'SVK', 'SWE', 'TUR']
Years: 2010 - 2024


,host_iso,year,indicator_code,value
0,AUT,2024,NY.GDP.MKTP.KD.ZG,-0.659090
1,AUT,2023,NY.GDP.MKTP.KD.ZG,-0.786244
2,AUT,2022,NY.GDP.MKTP.KD.ZG,5.330964
3,AUT,2021,NY.GDP.MKTP.KD.ZG,4.923092
4,AUT,2020,NY.GDP.MKTP.KD.ZG,-6.318255


In [10]:
wdi_wide = (
    wdi_tidy
    .pivot_table(
        index=["host_iso", "year"],
        columns="indicator_code",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)

wdi_wide = wdi_wide.rename(columns={
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",
    "FP.CPI.TOTL.ZG": "inflation",
    "SL.UEM.TOTL.ZS": "unemployment",
    "NE.TRD.GNFS.ZS": "trade_pct_gdp",
    "BN.CAB.XOKA.GD.ZS": "current_account_pct_gdp",
    "SP.POP.TOTL": "host_population",
})

numeric_cols = [
    "gdp_growth",
    "inflation",
    "unemployment",
    "trade_pct_gdp",
    "current_account_pct_gdp",
    "host_population",
]

for col in numeric_cols:
    if col in wdi_wide.columns:
        wdi_wide[col] = pd.to_numeric(wdi_wide[col], errors="coerce")

wdi_wide = wdi_wide.sort_values(["host_iso", "year"]).reset_index(drop=True)

print("WDI wide shape:", wdi_wide.shape)
print(wdi_wide.isna().sum())
wdi_wide.head()

WDI wide shape: (270, 8)
indicator_code
host_iso                   0
year                       0
current_account_pct_gdp    1
inflation                  0
trade_pct_gdp              1
gdp_growth                 1
unemployment               1
host_population            0
dtype: int64


indicator_code,host_iso,year,current_account_pct_gdp,inflation,trade_pct_gdp,gdp_growth,unemployment,host_population
0,AUT,2010,2.942949,1.813534,99.644883,1.808982,4.883,8363404.0
1,AUT,2011,1.585010,3.286579,105.775098,2.927468,4.637,8391643.0
2,AUT,2012,1.510000,2.485676,105.840791,0.628246,4.909,8429991.0
3,AUT,2013,1.752992,2.000156,104.985374,-0.250726,5.367,8479823.0
4,AUT,2014,2.313939,1.605812,104.496740,0.755799,5.674,8546356.0


## Panel Construction

This section combines the conflict, refugee, and macroeconomic data.

The panel structure is:

`conflict_case + host_country + year`

The same column structure is used for both Syria and Ukraine. This makes the later EDA and ML analysis cleaner.

In [11]:
def build_case_panel(case):
    origin_iso = case["origin_iso"]
    origin_name = case["origin_name"]
    conflict_case = case["conflict_case"]
    host_iso_list = list(case["hosts"].keys())

    panel_index = pd.MultiIndex.from_product(
        [host_iso_list, range(YEAR_FROM, YEAR_TO + 1)],
        names=["host_iso", "year"]
    )

    panel = pd.DataFrame(index=panel_index).reset_index()
    panel["origin_iso"] = origin_iso
    panel["origin_name"] = origin_name
    panel["conflict_case"] = conflict_case

    panel = panel.merge(
        unhcr_all,
        on=["conflict_case", "origin_iso", "host_iso", "year"],
        how="left"
    )

    panel = panel.merge(
        wdi_wide,
        on=["host_iso", "year"],
        how="left"
    )

    panel = panel.merge(
        conflict_intensity_all[["conflict_case", "origin_iso", "year", "conflict_intensity"]],
        on=["conflict_case", "origin_iso", "year"],
        how="left"
    )

    # If UNHCR does not return a host-year row, treat refugee and asylum counts as zero.
    panel["refugee_stock"] = panel["refugee_stock"].fillna(0)
    panel["asylum_seekers"] = panel["asylum_seekers"].fillna(0)

    panel["refugees_per_1000"] = np.where(
        (panel["host_population"].notna()) &
        (panel["host_population"] > 0),
        1000 * panel["refugee_stock"] / panel["host_population"],
        np.nan
    )

    panel["log_refugee_stock"] = np.log1p(panel["refugee_stock"])

    host_name_map = {iso: info["name"] for iso, info in case["hosts"].items()}
    host_group_map = {iso: info["group"] for iso, info in case["hosts"].items()}

    panel["host_name"] = panel["host_iso"].map(host_name_map)
    panel["host_group"] = panel["host_iso"].map(host_group_map)

    panel["post_conflict"] = (panel["year"] >= case["post_conflict_start"]).astype(int)

    panel = panel[[
        "conflict_case",
        "origin_name",
        "origin_iso",
        "host_name",
        "host_iso",
        "host_group",
        "year",
        "post_conflict",
        "conflict_intensity",
        "refugee_stock",
        "asylum_seekers",
        "host_population",
        "refugees_per_1000",
        "log_refugee_stock",
        "gdp_growth",
        "inflation",
        "unemployment",
        "trade_pct_gdp",
        "current_account_pct_gdp",
    ]].sort_values(["conflict_case", "host_iso", "year"]).reset_index(drop=True)

    return panel

In [12]:
case_panels = []

for case in CONFLICT_CASES:
    print("Building panel for:", case["conflict_case"])
    case_panel = build_case_panel(case)
    print(case_panel.shape)
    case_panels.append(case_panel)

combined_panel = pd.concat(case_panels, ignore_index=True)

print("Combined panel shape:", combined_panel.shape)
combined_panel.head()

Building panel for: Syrian Civil War
(135, 19)
Building panel for: Russia-Ukraine War
(195, 19)
Combined panel shape: (330, 19)


,conflict_case,origin_name,origin_iso,host_name,host_iso,host_group,year,post_conflict,conflict_intensity,refugee_stock,asylum_seekers,host_population,refugees_per_1000,log_refugee_stock,gdp_growth,inflation,unemployment,trade_pct_gdp,current_account_pct_gdp
0,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2010,0,0.0,505.0,471.0,8363404.0,0.060382,6.226537,1.808982,1.813534,4.883,99.644883,2.942949
1,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2011,1,1203.0,903.0,456.0,8391643.0,0.107607,6.806829,2.927468,3.286579,4.637,105.775098,1.585010
2,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2012,1,50490.0,1675.0,694.0,8429991.0,0.198695,7.424165,0.628246,2.485676,4.909,105.840791,1.510000
3,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2013,1,72016.0,2748.0,1561.0,8479823.0,0.324063,7.918992,-0.250726,2.000156,5.367,104.985374,1.752992
4,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2014,1,65345.0,6653.0,0.0,8546356.0,0.778460,8.802973,0.755799,1.605812,5.674,104.496740,2.313939


## Validation

This section checks whether the combined dataset has the expected structure.

Expected size:

- Syria: 9 host countries × 15 years = 135 rows
- Ukraine: 13 host countries × 15 years = 195 rows
- Total: 330 rows

In [13]:
print("Panel shape:", combined_panel.shape)

print("\nRows by conflict case:")
print(combined_panel.groupby("conflict_case")["year"].count())

print("\nRows by host group:")
print(combined_panel.groupby(["conflict_case", "host_group"])["year"].count())

print("\nYear range by conflict:")
print(combined_panel.groupby("conflict_case")["year"].agg(["min", "max", "count"]))

print("\nMissing values:")
print(combined_panel.isna().sum())

combined_panel.head(20)

Panel shape: (330, 19)

Rows by conflict case:
conflict_case
Russia-Ukraine War    195
Syrian Civil War      135
Name: year, dtype: int64

Rows by host group:
conflict_case       host_group                
Russia-Ukraine War  neighboring_host               75
                    non_neighbor_european_host    120
Syrian Civil War    neighboring_host               60
                    non_neighbor_european_host     60
                    regional_non_neighbor_host     15
Name: year, dtype: int64

Year range by conflict:
                     min   max  count
conflict_case                        
Russia-Ukraine War  2010  2024    195
Syrian Civil War    2010  2024    135

Missing values:
conflict_case              0
origin_name                0
origin_iso                 0
host_name                  0
host_iso                   0
host_group                 0
year                       0
post_conflict              0
conflict_intensity         0
refugee_stock              0
asylum_seekers 

,conflict_case,origin_name,origin_iso,host_name,host_iso,host_group,year,post_conflict,conflict_intensity,refugee_stock,asylum_seekers,host_population,refugees_per_1000,log_refugee_stock,gdp_growth,inflation,unemployment,trade_pct_gdp,current_account_pct_gdp
0,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2010,0,0.0,505.0,471.0,8363404.0,0.060382,6.226537,1.808982,1.813534,4.883,99.644883,2.942949
1,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2011,1,1203.0,903.0,456.0,8391643.0,0.107607,6.806829,2.927468,3.286579,4.637,105.775098,1.585010
2,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2012,1,50490.0,1675.0,694.0,8429991.0,0.198695,7.424165,0.628246,2.485676,4.909,105.840791,1.510000
3,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2013,1,72016.0,2748.0,1561.0,8479823.0,0.324063,7.918992,-0.250726,2.000156,5.367,104.985374,1.752992
4,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2014,1,65345.0,6653.0,0.0,8546356.0,0.778460,8.802973,0.755799,1.605812,5.674,104.496740,2.313939
5,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2015,1,50181.0,14894.0,18669.0,8642699.0,1.723304,9.608781,1.303523,0.896563,5.802,103.117582,1.588327
6,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2016,1,43650.0,30958.0,9812.0,8736668.0,3.543456,10.340419,2.117220,0.891592,6.064,101.573495,2.571488
7,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2017,1,24126.0,43888.0,3937.0,8797566.0,4.988653,10.689419,2.272250,2.081269,5.561,105.556067,1.244446
8,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2018,1,13072.0,49179.0,1538.0,8840521.0,5.562907,10.803242,2.484221,1.998380,4.933,108.377970,0.908177
9,Syrian Civil War,Syria,SYR,Austria,AUT,non_neighbor_european_host,2019,1,7379.0,51955.0,1304.0,8879920.0,5.850841,10.858152,1.754976,1.530896,4.560,108.207285,2.384230


## Save Output Files

This section saves the final combined panel and a missing-value summary.

The combined panel will be used in the next notebook for multi-conflict exploratory analysis and machine learning.

In [14]:
processed_path = PROCESSED_DIR / "conflict_host_panel_2010_2024.csv"

combined_panel.to_csv(processed_path, index=False)

print("Saved combined panel to:", processed_path)

Saved combined panel to: ../data/processed/conflict_host_panel_2010_2024.csv


In [15]:
missing_summary = combined_panel.isna().sum().reset_index()
missing_summary.columns = ["variable", "missing_count"]
missing_summary["missing_rate"] = missing_summary["missing_count"] / len(combined_panel)

missing_path = TABLE_DIR / "multi_conflict_missing_summary.csv"
missing_summary.to_csv(missing_path, index=False)

print("Saved missing summary to:", missing_path)

missing_summary

Saved missing summary to: ../outputs/tables/multi_conflict_missing_summary.csv


,variable,missing_count,missing_rate
0,conflict_case,0,0.00000
1,origin_name,0,0.00000
2,origin_iso,0,0.00000
3,host_name,0,0.00000
4,host_iso,0,0.00000
5,host_group,0,0.00000
6,year,0,0.00000
7,post_conflict,0,0.00000
8,conflict_intensity,0,0.00000
9,refugee_stock,0,0.00000


## Short Notes

This dataset is still exploratory.

It improves the project by adding a second conflict case and by allowing comparison between neighboring and non-neighbor host countries.

However, the two conflicts have different start years and different regional contexts. For this reason, the variable `post_conflict` is used separately for each conflict case instead of assuming a single common war year.